# Exercicis

Intenta utilitzar les funcions de `pandas` tant com puguis.

**No** facis servir cap `if` o `for` de Python.

**No** sobreescriguis la variable `df` enlloc (a no ser que es demani explicitament), mantén-la tal com està.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('../data/occupation.csv', sep='|')

## Dibuixa un histograma que mostri la distribució d'edats dels usuaris.

**Pista**: `kind='hist'`.

In [ ]:
df['age'].plot(figsize=(5,3), edgecolor = 'black', color='green', kind ='hist')
plt.title("Distribució d'edats dels usuaris.")
plt.xlabel("Edat")
plt.ylabel("Freqüència")
plt.show()

## Dibuixa un gràfic de sectors que mostri la distribució d'usuaris homes i dones.

In [ ]:
gènere_counts = df['gender'].value_counts()
fig, ax = plt.subplots(figsize=(3, 3), facecolor='black',edgecolor='green', linewidth=4)
ax.set_facecolor('skyblue')
ax.pie(gènere_counts, labels=gènere_counts.index, autopct='%1.1f%%', colors=['skyblue', 'lightgreen'], textprops={'color': 'white'})
plt.title("Distribució d'usuaris homes i dones.", color='white')
plt.show()

## Dibuixa un gràfic de barres de l'edat mitjana dels usuaris per a cada ocupació.

És a dir, mostra una barra per a cada ocupació, amb l'alçada essent l'edat mitjana de l'usuari.

**Extra**: Mostra-les ordenades descendentment, és a dir, l'ocupació amb l'edat mitjana més alta va primer i així successivament.

**Pista**: `kind='bar'` o `kind='barh'` (per mostrar-lo horitzontal).

In [ ]:
ocupacio_mitjana = df.groupby('occupation')['age'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4), facecolor='black',edgecolor='green', linewidth=4)
ax.set_facecolor('#222222')
ocupacio_mitjana.plot(kind='bar', color='#ffb347', edgecolor='white', ax=ax)
plt.title("Edat mitjana dels usuaris per a cada ocupació (Ordenat descendentment)", color='white', fontsize=12, pad=15)
plt.xlabel("Ocupació", color='white', fontsize=10)
plt.ylabel("Edat mitjana", color='white', fontsize=10)
ax.tick_params(colors='white')
ax.grid(axis='y', color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## Agrupa els individus per ocupació i mostra quants homes i dones hi ha a cada grup. Recorda utilitzar només `pandas`.

In [ ]:
grouped = df.groupby('occupation')['gender'].value_counts().to_frame(name='Quantity').reset_index()
grouped

## Afegeix aleatòriament valors `NaN` (`np.nan`) a la columna age. Després filtra el conjunt de dades, mostrant totes les files amb qualsevol valor `NaN`.

Aquí has de modificar la variable `df`.

In [ ]:
indexs_aleatoris = df.sample(10).index
df.loc[indexs_aleatoris, 'age'] = np.nan
files_amb_nan = df[df['age'].isna()]
files_amb_nan

## Reemplaça tots els valors `NaN` per `-1`.

No modifiquis `df`.

In [ ]:
reemplaçat = df.fillna(-1)
reemplaçat

## Reemplaça els valors `NaN` de la columna age per la mitjana de tots els valors de la columna.

Modifica `df`.

In [ ]:
mitjana_age = df['age'].mean()
df['age'] = df['age'].fillna(mitjana_age)
df

## Trams d'edat amb ocupacions

Crea un nou DataFrame on classifiquis els usuaris en trams d'edat: 18-30, 31-50, 51-70 i 70+.

Després, per a cada ocupació i tram d'edat, calcula l'edat mitjana i el percentatge d'homes.

**Pista**: Consulta el mètode `cut` de pandas per calcular els trams d'edat.

In [ ]:
limites = [18, 30, 50, 70, float('inf')]
etiquetas = ['18-30', '31-50', '51-70', '70+']
df['Tramo_Edad'] = pd.cut(df['age'], bins=limites, labels=etiquetas, right=True)
df['is_male'] = (df['gender'] == 'M').astype(int)
df_resum = df.groupby(['occupation', 'Tramo_Edad'], observed=False).agg(
    Edat_Mitjana= ('age', 'mean'),
    Percentatge_Homes=('is_male', lambda x: round(x.mean() * 100,2))
).reset_index()

df_resum['Edat_Mitjana'] = df_resum['Edat_Mitjana'].round(2)
df_resum

## Predomini de gènere per codi postal

Identifica els codis postals on un sol gènere representa més del 90% dels individus.

In [ ]:
proporciones = df.groupby('zip_code')['gender'].value_counts(normalize=True) * 100
predominios = proporciones[proporciones > 90].reset_index()
predominios

## Ocupacions més diverses

Identifica les 5 ocupacions principals que tenen la proporció home-dona més equilibrada.

**Pista**:
- Si fas un `groupby()` amb occupation i gender, mira què proporciona el mètode `size()`. Bàsicament, retorna una Sèrie "multiindex" amb dos índexs: occupation i gender, amb els valors corresponents al nombre d'elements de cada grup.
- Aquest multiindex es pot "desagrupar" i transformar en un DF amb occupation com a índex i gender com a columnes amb el mètode `unstack()`.
- A partir d'aquí, pensa com fer la resta.

In [21]:
# 1. Comptem quants homes i dones hi ha per ocupació i ho passem a columnes
df_counts = df.groupby(['occupation', 'gender']).size().unstack(fill_value=0)
df_counts['sumemHM'] = (df_counts['M']).astype(int) + (df_counts['F']).astype(int)
df_counts['propor_Homes'] =  (((df_counts['M']).astype(int)/df_counts['sumemHM'])*100).round(2)
df_counts['propor_equilibri'] = abs(df_counts['propor_Homes'] - 50)
df_counts.sort_values(by=['propor_equilibri'], ascending=True, inplace=True)
ocupacions_diverses = df_counts.head(5).reset_index()
ocupacions_diverses[['occupation', 'M', 'F', 'sumemHM', 'propor_Homes']]


gender,occupation,M,F,sumemHM,propor_Homes
0,artist,15,13,28,53.57
1,administrator,43,36,79,54.43
2,none,5,4,9,55.56
3,librarian,22,29,51,43.14
4,writer,26,19,45,57.78


## Densitat d'usuaris per estat

Suposant que els dos primers caràcters del `zip_code` representen un estat dels EUA, esbrina quin estat té la densitat d'usuaris més alta (nombre d'usuaris per ocupació única).

## Ocupacions rares

Identifica les ocupacions que són exclusives de cada gènere (és a dir, ocupacions exercides només per homes o només per dones).

## Proporció de gènere en sectors

Basant-te en els trams d'edat de l'exercici 2.1., per a cada tram d'edat dibuixa un gràfic de sectors de la proporció de gènere.

## Rang d'edat per ocupació

Per a cada ocupació, calcula el rang d'edat (màxim - mínim). Mostra les 10 ocupacions principals amb els rangs d'edat més diversos utilitzant un gràfic de barres horitzontal.